# 🎯 Learning Objectives

Understand how Machine Learning works.

Learn the ML workflow from data to prediction.


# 💻 Tasks

Install Scikit-learn.

Learn the Machine Learning workflow.

Split the dataset into training and testing sets.

Train a simple Linear Regression model.

Make predictions using the trained model.

Push your work to GitHub.

# What is Linear Regression?

Linear Regression is a supervised machine learning algorithm used to predict a continuous numerical value by identifying the linear relationship between input features and a target variable.
It tries to find the best-fitting straight line (or hyperplane when multiple features are used) that represents the relationship between the input variables and the target.

Examples:
- Predicting house prices based on location, income, and house size.
- Predicting salary based on years of experience.
- Predicting sales based on advertising expenditure.


# 2. How Linear Regression Works
Linear Regression works through the following steps:
1. Input data: Provide input features (X) and known target values (y).
2. Find the relationship: The algorithm estimates the relationship between the input features and the target.
3. Fit the best line: It learns coefficients and an intercept that minimize the sum of squared differences between actual and predicted values.
4. Make predictions: It uses the learned equation to predict target values for new data.
5. Evaluate performance: Metrics such as MAE, RMSE, and R² help measure how well the model performs on unseen data.

# ML workflow

01 - Data Collection
Gather data from CSV files, databases, APIs, etc.



02- Data Preprocessing
Handle missing values, duplicates, incorrect types and outliers.



03- Exploratory Data Analysis (EDA)
Understand distributions, patterns and relationships.



04- Feature Engineering
Create useful input variables for the model.



05 - Train-Test Split
Separate data for training and unbiased evaluation.



06- Model Selection & Training
Choose an algorithm and fit it to the training data.



07-Model Evaluation
Measure performance using appropriate metrics.



08-Prediction & Deployment
Predict outcomes for new data and integrate the model into an application.

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter


file_path = "housing.csv"


df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "camnugent/california-housing-prices",
  file_path,

)

print("First 5 records:", df.head())

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df = df.dropna().reset_index(drop=True)

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
encoder = OneHotEncoder(sparse_output=False, drop=None)
encoded = encoder.fit_transform(df[['ocean_proximity']])

In [ ]:
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(['ocean_proximity']), index=df.index)
df = pd.concat([df.drop('ocean_proximity', axis=1), encoded_df], axis=1)

In [ ]:
df

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(df['median_house_value'])
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['median_house_value'], kde=True, bins=50)
plt.title('MedHouseVal Distribution with KDE')
plt.show()

In [ ]:
corr = df.corr()
print(corr['median_house_value'].sort_values(ascending=False))

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix — California Housing')
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
scatter = plt.scatter(
    df['longitude'], df['latitude'],
    c=df['median_house_value'],
    cmap='viridis',
    s=10,
    alpha=0.6
)
plt.colorbar(scatter, label='Median House Value')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Geographic Distribution of House Values')
plt.show()

In [ ]:
df

In [ ]:
df_modified = df.copy()

df_modified.loc[:, 'rooms_per_house'] = df_modified['total_rooms'] / df_modified['households']
df_modified.loc[:, 'bedrooms_per_room'] = df_modified['total_bedrooms'] / df_modified['total_rooms']
df_modified.loc[:, 'population_per_household'] = df_modified['population'] / df_modified['households']

print("First 5 records of df_modified with new features:")
display(df_modified.head())

# Replace infinite values from feature engineering with missing values
df_modified = df_modified.replace([np.inf, -np.inf], np.nan)


In [ ]:
df_modified=df_modified.drop(columns=['total_rooms','population','total_bedrooms','households'])

In [ ]:
df_modified

In [ ]:
x=df_modified.drop('median_house_value', axis=1)
y=df['median_house_value']

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.20,random_state=20)

In [ ]:
x_train

In [ ]:
x_test

In [ ]:
y_train

In [ ]:
y_test

In [ ]:
from sklearn.impute import SimpleImputer

# To Handle any missing values before scaling
imputer = SimpleImputer(strategy="median")
x_train_imputed = imputer.fit_transform(x_train)
x_test_imputed = imputer.transform(x_test)

sc = StandardScaler()
x_train_sc = sc.fit_transform(x_train_imputed)
x_test_sc = sc.transform(x_test_imputed)

In [ ]:
x_train_sc

In [ ]:
x_test_sc

In [ ]:
model = LinearRegression()
model.fit(x_train_sc, y_train)

In [ ]:
y_pred = model.predict(x_test_sc)

In [ ]:
from sklearn.metrics import mean_absolute_error

print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.2, s=10)

plt.plot([0, 500000], [0, 500000], 'r--', label="Perfect Prediction")
plt.xlim(0, 520000)
plt.ylim(0, 520000)

plt.xlabel("Actual House Prices")
plt.ylabel("Predicted House Prices")
plt.title("Actual vs Predicted House Prices")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
comparison = pd.DataFrame({
    "Actual Price": y_test.values[:10],
    "Predicted Price": y_pred[:10]
})

comparison["Difference"] = comparison["Actual Price"] - comparison["Predicted Price"]
comparison["Absolute Difference"] = comparison["Difference"].abs()

print(comparison.round(2))